# 13 · Stop reasons and API errors
Level L2 · Part 2 · core · ~25 min · builds on 07

## Goal
Write `classify(resp)`, which sorts any Gemini response into `ok`, `cut`, `blocked`, `bad_tool_call`
or `empty`, and `safe_call`, which turns a 4xx or 5xx into a logged status plus your own call id.
Know the Anthropic names for the same endings.

## The idea
A call can end in two ways. Either the server answers, and the **stop reason** says why the model stopped
writing. Or the server refuses the request, and the SDK raises an **API error** with an HTTP status.
Only one stop reason means "finished normally". The others mean the text is cut, missing or not safe
to use, even when it looks fine. Errors split into "your request is wrong" (4xx: fix it, never retry)
and "the server had a problem" (5xx: retry, lesson 14). Every failure needs an id in the log, or you
cannot find it again.

Analogy: a parcel arrives with a delivery note. "Delivered", "partial delivery" and "refused at customs"
all arrive as a box. You read the note before you open it.

## Picture
```mermaid
flowchart TD
    C["safe_call(prompt)"] -->|"HTTP 4xx / 5xx"| E["APIError: log status + call id"]
    C -->|"HTTP 200"| R["response"]
    R --> K{"classify"}
    K -->|STOP + text| OK["ok: use it"]
    K -->|MAX_TOKENS| CUT["cut: more tokens or retry"]
    K -->|"SAFETY / block"| BL["blocked: do not retry as is"]
    K -->|MALFORMED_FUNCTION_CALL| TC["bad_tool_call: never run it"]
    K -->|no text| EM["empty: retry once"]
```

## Step by step
### 1. Start the recorder and set up
Same setup as lesson 07: the recorder, Scout's notes, the pinned model id and one client. The config
helper `cfg` builds a `GenerateContentConfig` with automatic function calling switched off.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("13_stop-reasons-and-errors")

LLM mode: replay (13_stop-reasons-and-errors.yaml)


Now the client, plus a small helper so every call below states only what changes.

In [2]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types, errors

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")

def cfg(**kwargs) -> types.GenerateContentConfig:
    off = types.AutomaticFunctionCallingConfig(disable=True)
    return types.GenerateContentConfig(automatic_function_calling=off, **kwargs)

print("model:", MODEL)

model: gemini-3.5-flash-lite


Setup from lesson 07: Scout's notes and the summary prompt for note 5.

In [3]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
note = NOTES[4]
prompt = f"Summarise this note in one short sentence.\nNote {note['id']}: {note['text']}"
print(prompt)

Summarise this note in one short sentence.
Note 5: The rover Perseverance landed on Mars in February 2021.


### 2. The normal ending: `STOP`
With room to finish, the model ends by choice. Gemini calls this finish reason `STOP`; Anthropic calls
it `end_turn`. This is the only case where the text is complete.

In [4]:
ok = client.models.generate_content(model=MODEL, contents=prompt, config=cfg(max_output_tokens=60))
print(ok.candidates[0].finish_reason.value, "|", ok.text)

STOP | Perseverance landed on Mars in February 2021.


### 3. Forced cut-off: `MAX_TOKENS`
Now the same Scout summary with only 8 output tokens. The model is stopped mid-sentence. This is
**max_tokens truncation**: the text is real, but unfinished.

In [5]:
cut = client.models.generate_content(model=MODEL, contents=prompt, config=cfg(max_output_tokens=8))
print(cut.candidates[0].finish_reason.value, "|", repr(cut.text))
print("output tokens:", cut.usage_metadata.candidates_token_count)

MAX_TOKENS | 'The rover Perseverance landed'
output tokens: 4


The output count is below 8 (lesson 07 saw the same), so counting tokens cannot detect the cut. Only the
finish reason can.

### 4. Stop sequences: stopping on purpose
A **stop sequence** (lesson 02) is text that ends the reply when the model writes it; the sequence itself
is left out. Here we ask for three Mars facts and stop at `"3."`, so we get two.

In [6]:
ask = "List three facts about Mars from these notes, on one line, as '1. ... 2. ... 3. ...'.\n"
ask += "\n".join(n["text"] for n in NOTES if "Mars" in n["text"])
seq = client.models.generate_content(model=MODEL, contents=ask,
                                     config=cfg(max_output_tokens=80, stop_sequences=["3."]))
print(seq.candidates[0].finish_reason.value, "|", repr(seq.text))

STOP | '1. Mars has two small moons, Phobos and Deimos. 2. A day on Mars lasts about 24 hours and 39 minutes. '


Gemini reports plain `STOP`: from the finish reason alone you cannot tell "finished" from "hit my
stop sequence". Anthropic is more precise: `stop_reason` is `"stop_sequence"` and a `stop_sequence`
field says which one fired. With Gemini, if it matters, check the text yourself.

### 5. A tool call cut off mid-JSON
A **tool call** is a JSON request from the model asking your code to run a function. Lesson 22 teaches
them; here we only need to see one get cut. We declare a `save_note` function, ask for a long note and
give the model 15 tokens. Read the output, not the declaration.

In [7]:
save_note = types.FunctionDeclaration(
    name="save_note", description="Save a note to Scout's notes.",
    parameters_json_schema={"type": "object", "properties": {"text": {"type": "string"}}, "required": ["text"]})
tool_cut = client.models.generate_content(
    model=MODEL, contents="Use save_note to save an 80-word note about the Perseverance rover.",
    config=cfg(tools=[types.Tool(function_declarations=[save_note])], max_output_tokens=15))
cand = tool_cut.candidates[0]
print(cand.finish_reason.value, "| parts:", cand.content.parts, "| text:", tool_cut.text, "| calls:", tool_cut.function_calls)

MALFORMED_FUNCTION_CALL | parts: None | text: None | calls: None


Gemini does not hand back half a JSON object: it drops the call and reports `MALFORMED_FUNCTION_CALL`,
with no parts at all. (Anthropic instead returns `stop_reason: "max_tokens"` with a half-filled
`tool_use` block.) The rule is the same for both: **never run a cut-off tool call**; retry with a bigger
`max_tokens`.

### 6. Safety blocks and refusals
Sometimes the model will not answer. Gemini has two places for this. A **safety block** on the prompt
sets `prompt_feedback.block_reason` and returns no candidates at all. A block on the reply sets the
candidate's finish reason to `SAFETY` (or `PROHIBITED_CONTENT`, `BLOCKLIST`, `SPII`, `RECITATION`).
Scout's notes never trigger these, so we build the two shapes by hand with the SDK's own types. These
are hand-built objects, not model output.

In [8]:
blocked_prompt = types.GenerateContentResponse(
    prompt_feedback=types.GenerateContentResponsePromptFeedback(block_reason="PROHIBITED_CONTENT"))
blocked_reply = types.GenerateContentResponse(candidates=[types.Candidate(
    finish_reason="SAFETY", content=types.Content(role="model", parts=[]))])
print("prompt block:", blocked_prompt.prompt_feedback.block_reason.value, "| candidates:", blocked_prompt.candidates, "| text:", blocked_prompt.text)
print("reply block :", blocked_reply.candidates[0].finish_reason.value, "| text:", blocked_reply.text)

prompt block: PROHIBITED_CONTENT | candidates: None | text: None
reply block : SAFETY | text: None


In both cases `.text` is `None`, not an empty string. Anthropic signals a **refusal** with
`stop_reason: "refusal"`; the docs say to read `stop_details` and retry on a fallback model, not to
resend the same request.

### 7. Reference: Anthropic names
Scout runs on Gemini, so only the Gemini column is what you ran above. The Anthropic column is for
reading other code and docs; it is not run here. Two rows have no Gemini twin:
- `pause_turn`: a tool running on Anthropic's servers paused; send the reply back (lesson 44).
- `model_context_window_exceeded`: the conversation filled the context window; treat it as cut.

In [9]:
ENDINGS = [  # (Anthropic stop_reason, Gemini finish_reason, what to do)
    ("end_turn", "STOP", "use the text (check it is not empty)"),
    ("max_tokens", "MAX_TOKENS", "text is cut: raise the limit or continue"),
    ("stop_sequence", "STOP", "expected stop: read which sequence fired"),
    ("tool_use", "STOP + function_calls", "run the tool (lesson 22)"),
    ("max_tokens + tool_use", "MALFORMED_FUNCTION_CALL", "never run it: retry with more tokens"),
    ("pause_turn", "(none)", "send the reply back to continue"),
    ("refusal", "SAFETY / prompt block", "do not resend as is; log and tell the user"),
    ("model_context_window_exceeded", "MAX_TOKENS", "treat as cut (lesson 05)"),
]
for a, g, todo in ENDINGS:
    print(f"{a:30} {g:26} {todo}")

end_turn                       STOP                       use the text (check it is not empty)
max_tokens                     MAX_TOKENS                 text is cut: raise the limit or continue
stop_sequence                  STOP                       expected stop: read which sequence fired
tool_use                       STOP + function_calls      run the tool (lesson 22)
max_tokens + tool_use          MALFORMED_FUNCTION_CALL    never run it: retry with more tokens
pause_turn                     (none)                     send the reply back to continue
refusal                        SAFETY / prompt block      do not resend as is; log and tell the user
model_context_window_exceeded  MAX_TOKENS                 treat as cut (lesson 05)


### 8. `classify`: one function for every ending
Scout needs one place that turns a response into a decision. Order matters: a blocked prompt has no
candidate, so check that first; an empty `STOP` is still a failure.

In [10]:
BLOCK_REASONS = {"SAFETY", "PROHIBITED_CONTENT", "BLOCKLIST", "SPII", "RECITATION"}

def classify(resp) -> str:
    if resp.prompt_feedback and resp.prompt_feedback.block_reason:
        return "blocked"
    if not resp.candidates:
        return "empty"
    reason = resp.candidates[0].finish_reason.value
    if reason == "MAX_TOKENS":
        return "cut"
    if reason in BLOCK_REASONS:
        return "blocked"
    if reason == "MALFORMED_FUNCTION_CALL":
        return "bad_tool_call"
    if reason == "STOP" and (resp.text or resp.function_calls):
        return "ok"
    return "empty"

Run it on every response from this lesson, plus a hand-built empty one (no candidates, no block reason).

In [11]:
empty = types.GenerateContentResponse(candidates=[])
for name, r in [("ok", ok), ("cut", cut), ("seq", seq), ("tool_cut", tool_cut),
                ("blocked_prompt", blocked_prompt), ("blocked_reply", blocked_reply), ("empty", empty)]:
    print(f"{name:15} -> {classify(r)}")

ok              -> ok
cut             -> cut
seq             -> ok
tool_cut        -> bad_tool_call
blocked_prompt  -> blocked
blocked_reply   -> blocked
empty           -> empty


The stop-sequence reply counts as `ok`: Scout asked for that stop.

### 9. When the request itself is wrong: a 400
Now the other way a call ends. We send a bad request: a negative `max_output_tokens`. The server rejects
it before the model runs, and the SDK raises `errors.ClientError` (all 4xx). `errors.ServerError` is the
5xx family; both inherit from `errors.APIError`.

In [12]:
try:
    client.models.generate_content(model=MODEL, contents=prompt, config=cfg(max_output_tokens=-5))
except errors.APIError as e:
    bad = e
print(type(bad).__name__, "| code:", bad.code, "| status:", bad.status)
print("message:", bad.message)

ClientError | code: 400 | status: INVALID_ARGUMENT
message: Unable to submit request because it has a maxOutputTokens value of -5 but the supported range is from 1 (inclusive) to 65537 (exclusive). Update the value and try again.


The message says exactly what to fix. Retrying would fail the same way every time.

### 10. Which id do you log?
Support teams ask for the provider's **request id**. Anthropic's SDK has one on every error
(`e.request_id`); that is not run here, because Scout uses Gemini. Gemini gives `response_id` on a
success. Does its error carry one?

In [13]:
print("success response_id:", ok.response_id)
print("error body         :", bad.details)
print("error headers      :", sorted(bad.response.headers) if bad.response else None)

success response_id: 7cLHapWSFZj-7bEPvdqD6A8
error body         : {'error': {'code': 400, 'message': 'Unable to submit request because it has a maxOutputTokens value of -5 but the supported range is from 1 (inclusive) to 65537 (exclusive). Update the value and try again.', 'status': 'INVALID_ARGUMENT'}}
error headers      : ['content-length', 'content-type', 'date', 'server', 'transfer-encoding', 'vary', 'x-content-type-options', 'x-frame-options', 'x-xss-protection']


No id in the error body or headers, so Scout logs its own **call id** for every request.

### 11. `safe_call`: errors and stop reasons in one wrapper
The wrapper logs one line per call: call id, outcome, and the provider's id when there is one. It
returns `(status, response)` and never crashes Scout on an API error. A 429 (rate limit) is also a
`ClientError`, but unlike a 400 it is worth retrying, so it gets its own label; lesson 14 retries it.

In [14]:
import uuid

def safe_call(contents, max_output_tokens=60, **config) -> tuple[str, object]:
    call_id = uuid.uuid4().hex[:12]
    try:
        resp = client.models.generate_content(
            model=MODEL, contents=contents, config=cfg(max_output_tokens=max_output_tokens, **config))
    except errors.APIError as e:
        kind = "retryable" if e.code == 429 or e.code >= 500 else "client_error"
        print(f"[{call_id}] {kind} {e.code} {e.status}: {e.message[:70]}")
        return kind, None
    status = classify(resp)
    print(f"[{call_id}] {status} response_id={resp.response_id}")
    return status, resp

Two calls through the wrapper: a good one and the bad request again. The call ids are random, so yours
will differ.

In [15]:
for budget in (60, -5):
    status, resp = safe_call(prompt, max_output_tokens=budget)
    print("  ->", status, "|", resp.text if status == "ok" else "no text")

[dc9e76326804] ok response_id=8MLHas-HIuWBr8kP09Or0Qs
  -> ok | The rover Perseverance successfully landed on Mars in February 2021.
[bf2857322b36] client_error 400 INVALID_ARGUMENT: Unable to submit request because it has a maxOutputTokens value of -5 
  -> client_error | no text


## At work
- **The stop reason is logged on every call** next to the request id, model and tokens, so "why was this
  answer half a sentence?" has an answer in the logs. → lesson 49
- **4xx and 5xx are handled differently.** 400s are bugs to fix and must not be retried; 429 and 5xx are
  retried with backoff, and Anthropic's spend-cap 429 (a monthly spend limit: retrying is futile) is the exception that must not be. → lesson 14
- **Truncated tool calls are a known agent bug.** A loop that runs a half-written tool call corrupts data;
  production loops branch on the stop reason before touching tools. → lessons 26 and 30
- **Refusals fall back, not retry.** Teams route a refusal or a provider outage to another model or a
  polite message instead of resending the same request. → lessons 15 and 57

## What just happened
- A normal call ended with `STOP`; 8 output tokens gave `MAX_TOKENS` and a cut sentence.
- A stop sequence `"3."` ended the list after two facts, but Gemini still said `STOP`.
- A tool call cut at 15 tokens came back as `MALFORMED_FUNCTION_CALL` with no parts.
- Safety blocks (hand-built) have `.text` equal to `None`; a prompt block has no candidates.
- `classify` turned all seven responses into `ok`, `cut`, `blocked`, `bad_tool_call` or `empty`.
- A negative `max_output_tokens` raised `ClientError` 400 `INVALID_ARGUMENT`, with no request id, so
  `safe_call` logs its own call id.

## Common mistakes
**1. Treating `.text` as always a string.** On a blocked or empty response it is `None`, and string
methods crash.

In [16]:
print(blocked_reply.text.strip())

AttributeError: 'NoneType' object has no attribute 'strip'

Fix: classify first, and only read the text when the status is `ok`.

In [17]:
print(blocked_reply.text.strip() if classify(blocked_reply) == "ok" else f"no text ({classify(blocked_reply)})")

no text (blocked)


**2. Catching only server errors.** `ServerError` covers 5xx. A 400 is a `ClientError`, so it slips past.

In [18]:
try:
    raise bad   # re-raise the 400 we caught in step 9, without another call
except errors.ServerError:
    print("handled")

ClientError: 400 INVALID_ARGUMENT. {'error': {'code': 400, 'message': 'Unable to submit request because it has a maxOutputTokens value of -5 but the supported range is from 1 (inclusive) to 65537 (exclusive). Update the value and try again.', 'status': 'INVALID_ARGUMENT'}}

Fix: catch `errors.APIError` (or both classes) and branch on `e.code`, as `safe_call` does.

In [19]:
try:
    raise bad
except errors.APIError as e:
    print("handled:", e.code, "retry?", e.code >= 500 or e.code == 429)

handled: 400 retry? False


## Try it
Write `summarise(note_text, budget, call=safe_call)`. It sends the summary prompt with `call(prompt,
budget)`. If the status is `cut`, it retries **once** with four times the budget. It returns the text, or
`None` if the call still fails.

Test it with the stub `fake_call` below first. It needs no key: it returns `cut` when the budget is
under 20 and the `ok` reply from step 2 otherwise. So `summarise(note["text"], 8, call=fake_call)` should
print a sentence, and `budget=2` should give `None` (2 x 4 = 8 is still under 20). If you have a key, then
try the real `safe_call` with `budget=8`. Real calls made here are not recorded.

In [20]:
def fake_call(prompt, budget):
    return ("cut", None) if budget < 20 else ("ok", ok)

# your code here

Stop the recorder.

In [21]:
llm_replay.stop()

## Key terms
- **stop reason** — why a reply ended (`end_turn`, `max_tokens`, `refusal`, ...); Gemini calls it `finish_reason`.
- **max_tokens truncation** — a reply stopped by the output limit; real text, but unfinished.
- **malformed function call** — Gemini's finish reason for a tool call it could not complete; nothing to run.
- **safety block** — the provider withholds a prompt or reply; `.text` is `None`.
- **refusal** — the model declines to answer; Anthropic's `stop_reason: "refusal"`.
- **pause_turn** — Anthropic stop reason: a server-run tool loop paused; send the reply back.
- **API error** — the server rejected the call; an exception with an HTTP status (4xx yours, 5xx theirs).
- **request id** — the provider's id for one call, for support and log searches.
- **call id** — an id your code makes per call, so failures are findable even without a request id.